In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
from huggingface_hub import login

from fuseqa import *

sre = SRE()

In [ ]:
# Loading the dataset
from datasets import load_dataset
#ds = load_dataset("MinaGabriel/popqa-with-retrieval-20")["test"]#.select(range(100))
ds = load_dataset("MinaGabriel/entityquestions-final-clean")["train"].select(range(250))
len(ds)

In [ ]:
# =========================
# IMPORTS
# =========================
import os
import re
import tqdm
from typing import List
from datasets import Dataset
from wtpsplit import WtP

# =========================
# LOAD SPLITTER
# =========================
wtp = WtP("wtp-canine-s-12l")

# =========================
# CONFIG
# =========================
TOP_K_SENTENCES = 5
DOCS_PER_QUERY = 2
MAX_WORDS = 40
MAX_DOC_TOKENS = 150

def truncate_doc(doc: str, max_tokens: int = MAX_DOC_TOKENS) -> str:
    words = doc.split()
    if len(words) <= max_tokens:
        return doc
    return " ".join(words[:max_tokens])

# =========================
# CLEAN FUNCTION
# =========================
def clean_doc(doc: str) -> str:
    if not doc:
        return ""

    doc = re.sub(r'^"\s*[^"]+\s*"\s*', '', doc)

    # remove titles like "George Rankin"
    doc = re.sub(r'^[A-Z][^\n]{0,80}\n', '', doc)

    # normalize structure
    doc = doc.replace("\n", " ")
    doc = re.sub(r'\s+', ' ', doc).strip()

    return doc

  
def split_docs_to_sentences(docs: List[str]) -> List[str]:
    sentences = []

    for doc in docs:
        doc = clean_doc(doc)
        if not doc:
            continue
 
        try:
            splits = wtp.split(doc, lang_code="en")
        except:
            splits = []
 
        if not splits or len(splits) <= 1:
            splits = re.split(r'(?<=[.!?])\s+', doc)

        for s in splits:
            s = s.strip().rstrip(" .")
            words = s.split()

            if len(words) < 4:
                continue

            if len(words) > MAX_WORDS:
                s = " ".join(words[:MAX_WORDS])

            sentences.append(s)

    # dedupe
    return list(dict.fromkeys(sentences))


# =========================
# MAIN LOOP (STREAMING)
# =========================
new_rows = []

pbar = tqdm.tqdm(
    total=len(ds),
    desc="SRE (streaming, fixed split)",
    dynamic_ncols=True
)

for i in range(len(ds)):
    ex = {k: ds[k][i] for k in ds.column_names}

    q = ex["question"]
 
    raw_docs = ex.get("retrieved_docs") or []

    retrieved = [
        truncate_doc(d["text"] if isinstance(d, dict) else d)
        for d in raw_docs[:DOCS_PER_QUERY]
    ]
 
    sentences = split_docs_to_sentences(retrieved)
    print(f"{sentences}")
    if not sentences:
        continue

    # =========================
    # CONTEXT
    # =========================
    context = "\n".join([clean_doc(doc)[:250] for doc in retrieved])

    # =========================
    # SRE SCORING
    # =========================
    results = sre.score_sentences_batch(
        question=q,
        full_context=context,
        sentences=sentences,
        max_length=1024,
        batch_size=16
    )
 
    results = sorted(results, key=lambda x: x["score"], reverse=True)

    # =========================
    # TOP-K
    # =========================
    top_k = results[:TOP_K_SENTENCES]

    # =========================
    # FORMAT
    # =========================
    formatted_retrieved = [
        {
            "text": r["sentence"],
            "score": float(r["score"])
        }
        for r in top_k
    ]

    new_ex = dict(ex)
    new_ex["retrieved_docs"] = formatted_retrieved

    new_rows.append(new_ex)

    pbar.update(1)

pbar.close()

# =========================
# CREATE DATASET
# =========================
new_ds = Dataset.from_list(new_rows)

print("\n Sample:")
print(new_ds[0]["retrieved_docs"])



In [ ]:
# =========================
# SAVE
# =========================
save_path = os.path.expanduser("./entityquestions-sre-top5")
new_ds.save_to_disk(save_path)

print(f"\n Saved to: {save_path}")

In [ ]:
import json
rows = new_ds.select(range(50)).to_list()
print(json.dumps(rows, indent=2, ensure_ascii=False))

In [ ]:
# from datasets import load_from_disk

# # =========================
# # CONFIG
# # =========================
# LOCAL_PATH = "./entityquestions-sre-top5"   #   saved dataset
# HF_DATASET_NAME = "MinaGabriel/entityquestions-sre-top5" 
# PRIVATE = False   

# # =========================
# # LOAD LOCAL DATASET
# # =========================
# ds = load_from_disk(LOCAL_PATH)

# print(" Loaded dataset")
# print(ds)

# # =========================
# # OPTIONAL: sanity check
# # =========================
# print("\nSample:")
# print(ds[0]["retrieved_docs"])

# # =========================
# # PUSH TO HUB
# # =========================
# ds.push_to_hub(
#     HF_DATASET_NAME,
#     private=PRIVATE
# )

# print(f"\n Dataset pushed to: https://huggingface.co/datasets/{HF_DATASET_NAME}")